# Phase 3 — Alpha / diffusion_steps ablation

Budget: ~2 GPU-hr (sweep) + ~2.5 GPU-hr (replication of the winner)

Sweep pass (single seed=42, 300 steps):
- target_alpha ∈ {0.02, 0.05, 0.10, 0.20} at fixed diffusion_steps=3
- diffusion_steps ∈ {1, 2, 4} at whichever alpha looked most promising

Replication: whichever config wins the sweep, rerun Phase 2's 5-seed protocol on it.

## 0. Install & Setup
Install TorchDire from git and required dependencies, and pin the GPU to avoid DataParallel/bitsandbytes crashes.

In [ ]:
!pip install git+https://github.com/rajboopathiking/TorchDire.git
!pip install -q transformers datasets peft trl bitsandbytes accelerate scipy scipy

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 1. Configuration
Editable cell with all hyperparameters.

In [ ]:
MODEL_ID = "42dot/42dot_LLM-SFT-1.3B"
DATASET_ID = "arbml/alpagasus_cleaned"
MAX_STEPS = 300
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
OUTPUT_DIR = "/kaggle/working/phase3_output"

# Sweep config
ALPHA_VALUES = [0.02, 0.05, 0.10, 0.20]
FIXED_DIFFUSION_STEPS = 3
DIFFUSION_STEPS_VALUES = [1, 2, 4]
SWEEP_SEED = 42

# Replication config
REPLICATION_SEEDS = [42, 43, 44, 45, 46]

import warnings
warnings.filterwarnings("ignore")

## 2. Helper Functions
Functions to format examples, create SFT configurations, and run single experimental arms.

In [ ]:
import gc
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from torchdire import patch_llama_with_qgfd, register_qgfd_step_callback

def format_example(example):
    if "input" in example and example["input"]:
        return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

def run_single_arm(run_name, seed, target_alpha, diffusion_steps, use_qgfd=True, max_steps=MAX_STEPS):
    print(f"\n{'='*50}")
    print(f"Starting run: {run_name} (Seed: {seed})")
    print(f"QGFD: {use_qgfd}, Alpha: {target_alpha}, Diffusion Steps: {diffusion_steps}")
    print(f"{'='*50}")
    
    set_seed(seed)
    
    # Clean memory
    gc.collect()
    torch.cuda.empty_cache()
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if getattr(tokenizer, "pad_token", None) is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    
    if use_qgfd:
        patch_llama_with_qgfd(model, target_alpha=target_alpha, diffusion_steps=diffusion_steps)
        register_qgfd_step_callback(model, warmup_steps=50, total_steps=max_steps)
        
    dataset = load_dataset(DATASET_ID, split="train")
    
    sft_config = SFTConfig(
        output_dir=f"{OUTPUT_DIR}/{run_name}",
        max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        save_steps=1000,
        bf16=True,
        gradient_checkpointing=True,
        max_seq_length=512,
        packing=False,
        dataset_text_field="text"
    )
    
    def format_dataset(examples):
        return {"text": [format_example(dict(zip(examples, t))) for t in zip(*examples.values())]}
    formatted_dataset = dataset.map(format_dataset, batched=True, remove_columns=dataset.column_names)

    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        args=sft_config,
        tokenizer=tokenizer
    )
    
    trainer.train()
    
    log_history = trainer.state.log_history
    final_loss = None
    for entry in reversed(log_history):
        if "loss" in entry:
            final_loss = entry["loss"]
            break
            
    # Cleanup
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return final_loss

## 3. Alpha Sweep
Loop over alpha values at fixed diffusion_steps=3, seed=42, 300 steps.

In [ ]:
sweep_results_file = "/kaggle/working/phase3_alpha_sweep.csv"

alpha_sweep_results = []
if os.path.exists(sweep_results_file):
    print("Loading existing alpha sweep results...")
    df_alpha = pd.read_csv(sweep_results_file)
    alpha_sweep_results = df_alpha.to_dict('records')

completed_alphas = [r['target_alpha'] for r in alpha_sweep_results]

for alpha in ALPHA_VALUES:
    if alpha in completed_alphas:
        print(f"Skipping alpha={alpha}, already completed.")
        continue
        
    loss = run_single_arm(
        run_name=f"sweep_alpha_{alpha}",
        seed=SWEEP_SEED,
        target_alpha=alpha,
        diffusion_steps=FIXED_DIFFUSION_STEPS,
        use_qgfd=True
    )
    
    alpha_sweep_results.append({
        'target_alpha': alpha,
        'diffusion_steps': FIXED_DIFFUSION_STEPS,
        'final_loss': loss
    })
    
    # Incremental save
    pd.DataFrame(alpha_sweep_results).to_csv(sweep_results_file, index=False)

df_alpha = pd.DataFrame(alpha_sweep_results)
print("Alpha Sweep Results:")
display(df_alpha)

## 4. Pick Best Alpha
Analyze results to select the best alpha for the next sweep phase.

In [ ]:
best_alpha_row = df_alpha.loc[df_alpha['final_loss'].idxmin()]
best_alpha = best_alpha_row['target_alpha']
print(f"Best Target Alpha from Sweep: {best_alpha} (Loss: {best_alpha_row['final_loss']:.4f})")

## 5. Diffusion Steps Sweep
Loop over diffusion_steps at best alpha, seed=42, 300 steps.

In [ ]:
diff_sweep_results_file = "/kaggle/working/phase3_diff_sweep.csv"

diff_sweep_results = []
if os.path.exists(diff_sweep_results_file):
    print("Loading existing diffusion steps sweep results...")
    df_diff = pd.read_csv(diff_sweep_results_file)
    diff_sweep_results = df_diff.to_dict('records')
else:
    # add the FIXED_DIFFUSION_STEPS result from the alpha sweep so we don't rerun it
    diff_sweep_results.append({
        'target_alpha': best_alpha,
        'diffusion_steps': FIXED_DIFFUSION_STEPS,
        'final_loss': best_alpha_row['final_loss']
    })
    pd.DataFrame(diff_sweep_results).to_csv(diff_sweep_results_file, index=False)

completed_diff_steps = [r['diffusion_steps'] for r in diff_sweep_results]

for diff_steps in DIFFUSION_STEPS_VALUES:
    if diff_steps in completed_diff_steps:
        print(f"Skipping diffusion_steps={diff_steps}, already completed.")
        continue
        
    loss = run_single_arm(
        run_name=f"sweep_diff_{diff_steps}",
        seed=SWEEP_SEED,
        target_alpha=best_alpha,
        diffusion_steps=diff_steps,
        use_qgfd=True
    )
    
    diff_sweep_results.append({
        'target_alpha': best_alpha,
        'diffusion_steps': diff_steps,
        'final_loss': loss
    })
    
    # Incremental save
    pd.DataFrame(diff_sweep_results).to_csv(diff_sweep_results_file, index=False)

df_diff = pd.DataFrame(diff_sweep_results).sort_values('diffusion_steps')
print("Diffusion Steps Sweep Results:")
display(df_diff)

## 6. Pick Best Config
Analyze combined results to find the overall best configuration.

In [ ]:
best_overall_row = df_diff.loc[df_diff['final_loss'].idxmin()]
best_overall_alpha = best_overall_row['target_alpha']
best_overall_diff_steps = int(best_overall_row['diffusion_steps'])
print(f"Overall Best Configuration: Alpha={best_overall_alpha}, Diffusion Steps={best_overall_diff_steps} (Loss: {best_overall_row['final_loss']:.4f})")

## 7. 5-Seed Replication of Winner
Run the winning configuration and a standard baseline across 5 seeds with paired statistical analysis.

In [ ]:
replication_results_file = "/kaggle/working/phase3_replication.csv"

replication_results = []
if os.path.exists(replication_results_file):
    print("Loading existing replication results...")
    df_repl = pd.read_csv(replication_results_file)
    replication_results = df_repl.to_dict('records')

for seed in REPLICATION_SEEDS:
    for use_qgfd in [False, True]:
        # Check if already done
        if any(r['seed'] == seed and r['use_qgfd'] == use_qgfd for r in replication_results):
            print(f"Skipping Seed={seed}, QGFD={use_qgfd} - already completed.")
            continue
            
        mode = "QGFD" if use_qgfd else "Baseline"
        run_name = f"replication_{mode.lower()}_seed_{seed}"
        
        alpha_to_use = best_overall_alpha if use_qgfd else 0.0
        diff_steps_to_use = best_overall_diff_steps if use_qgfd else 0
        
        loss = run_single_arm(
            run_name=run_name,
            seed=seed,
            target_alpha=alpha_to_use,
            diffusion_steps=diff_steps_to_use,
            use_qgfd=use_qgfd
        )
        
        replication_results.append({
            'seed': seed,
            'use_qgfd': use_qgfd,
            'target_alpha': alpha_to_use,
            'diffusion_steps': diff_steps_to_use,
            'final_loss': loss
        })
        
        # Incremental save
        pd.DataFrame(replication_results).to_csv(replication_results_file, index=False)

df_repl = pd.DataFrame(replication_results)
print("Replication Results:")
display(df_repl)

## 8. Statistical Analysis
Perform paired t-test, Wilcoxon signed-rank test, and 95% CI on the 5-seed replication results.

In [ ]:
from scipy import stats
import numpy as np

if not df_repl.empty:
    baseline_losses = df_repl[df_repl['use_qgfd'] == False].sort_values('seed')['final_loss'].values
    qgfd_losses = df_repl[df_repl['use_qgfd'] == True].sort_values('seed')['final_loss'].values
    
    if len(baseline_losses) == len(qgfd_losses) and len(baseline_losses) > 1:
        diffs = baseline_losses - qgfd_losses
        
        mean_diff = np.mean(diffs)
        std_diff = np.std(diffs, ddof=1)
        n = len(diffs)
        
        # Paired t-test
        t_stat, p_val_t = stats.ttest_rel(baseline_losses, qgfd_losses)
        
        # Wilcoxon
        w_stat, p_val_w = stats.wilcoxon(baseline_losses, qgfd_losses)
        
        # 95% CI
        ci = stats.t.interval(0.95, n-1, loc=mean_diff, scale=std_diff/np.sqrt(n))
        
        print(f"Mean Difference (Baseline - QGFD): {mean_diff:.5f}")
        print(f"95% CI of Difference: ({ci[0]:.5f}, {ci[1]:.5f})")
        print(f"Paired t-test p-value: {p_val_t:.5f}")
        print(f"Wilcoxon p-value: {p_val_w:.5f}")
        
        # Save stats
        with open("/kaggle/working/phase3_stats.txt", "w") as f:
            f.write(f"Mean Diff: {mean_diff}\nCI: {ci}\nT-test p: {p_val_t}\nWilcoxon p: {p_val_w}\n")
    else:
        print("Not all seeds have completed both Baseline and QGFD runs.")

## 9. Summary Plots
Bar charts of ablation results and paired difference plots for replication.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Alpha Sweep
if not df_alpha.empty:
    axes[0].plot(df_alpha['target_alpha'], df_alpha['final_loss'], marker='o', linestyle='-', color='b')
    axes[0].set_title(f'Alpha Sweep (Fixed diff_steps={FIXED_DIFFUSION_STEPS})')
    axes[0].set_xlabel('Target Alpha')
    axes[0].set_ylabel('Final Loss')
    axes[0].grid(True)

# Plot 2: Diffusion Steps Sweep
if not df_diff.empty:
    axes[1].plot(df_diff['diffusion_steps'], df_diff['final_loss'], marker='o', linestyle='-', color='g')
    axes[1].set_title(f'Diffusion Steps Sweep (Fixed alpha={best_alpha})')
    axes[1].set_xlabel('Diffusion Steps')
    axes[1].set_ylabel('Final Loss')
    axes[1].grid(True)

# Plot 3: Paired Difference
if not df_repl.empty and 'diffs' in locals():
    axes[2].bar(range(1, len(diffs)+1), diffs, color=['green' if d > 0 else 'red' for d in diffs])
    axes[2].set_title('Replication: Baseline Loss - QGFD Loss')
    axes[2].set_xlabel('Seed Iteration')
    axes[2].set_ylabel('Loss Difference (Positive = QGFD Better)')
    axes[2].axhline(0, color='black', linewidth=1)

plt.tight_layout()
plt.savefig("/kaggle/working/phase3_summary_plots.png")
plt.show()

## 10. Final Summary
Review the saved data at `/kaggle/working/phase3_*.csv` and the summary plot `phase3_summary_plots.png`.